# Person-in-WiFi, Stage 4: full training on Wi-Pose

This notebook runs the real training: it downloads the Wi-Pose dataset (1.4 GB), extracts and packs it, trains the network for 20 epochs, and reports PCK@0.2 on the official test split.

**Before running:** Runtime > Change runtime type > **GPU** (a T4 is fine). Expect roughly 5 to 10 minutes per epoch, 2 to 3 hours total.

**Do not** `pip install -r requirements.txt` here: Colab already ships torch with CUDA, and the repo pins the CPU build for local work. Everything this notebook needs is preinstalled.

**Disconnects:** checkpoints are written to your Google Drive every epoch. If the session dies, reconnect, rerun the cells from the top (download, extract, and pack all skip if their output already exists), and set `resume` in the training cell to the last saved checkpoint.

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.isdir("/content/Person-in-wifi"):
    !git clone https://github.com/adityagit94/Person-in-wifi.git /content/Person-in-wifi
%cd /content/Person-in-wifi
!git pull

Mount Drive so checkpoints survive a disconnect. If you skip this, checkpoints stay on the VM and vanish with the session.

In [ ]:
CKPT = "/content/checkpoints"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT = "/content/drive/MyDrive/wipose_checkpoints"
except Exception as e:
    print("Drive not mounted, checkpoints stay on the VM:", e)
os.makedirs(CKPT, exist_ok=True)
print("checkpoints ->", CKPT)

Download the dataset to the VM's local disk and extract it there (never onto mounted Drive: 166,600 small files on Drive is painfully slow). Then pack the files into contiguous arrays; per-sample HDF5 opens are the training bottleneck otherwise. Packing takes a few minutes and prints the keypoint coordinate maxima, which should sit just under the 480 x 640 source-frame estimate.

In [ ]:
if not os.path.exists("/content/Wi-Pose.rar"):
    !gdown 1WL6bJ-rSVdsclRt9RFc0l5hhtXYmfNg9 -O /content/Wi-Pose.rar
!ls -lh /content/Wi-Pose.rar

In [ ]:
!apt-get -qq install -y libarchive-tools > /dev/null
if not os.path.isdir("/content/wipose_raw/Wi-Pose/Train"):
    os.makedirs("/content/wipose_raw", exist_ok=True)
    !bsdtar -xf /content/Wi-Pose.rar -C /content/wipose_raw
!ls /content/wipose_raw/Wi-Pose

In [ ]:
if not os.path.exists("/content/packed/Train/csi.npy"):
    !python -m piw.pack --src /content/wipose_raw/Wi-Pose --dst /content/packed
else:
    print("already packed")

Sanity check: split sizes must match the official 132,847 / 33,753, and a rendered target should show an upright person with the skeleton on the blobs.

In [ ]:
import matplotlib.pyplot as plt
from piw.dataset import PackedWiPose
from piw.skeleton import LIMBS, NUM_JOINTS

train_ds = PackedWiPose("/content/packed", "Train")
test_ds = PackedWiPose("/content/packed", "Test")
print(f"train {len(train_ds)}  test {len(test_ds)}")
assert len(train_ds) == 132847 and len(test_ds) == 33753

it = train_ds[0]
kp = it["keypoints_grid"].numpy()
ok = kp[:, 2] >= 0.2
plt.figure(figsize=(7, 4))
plt.imshow(it["jhm"][:NUM_JOINTS].max(0), cmap="magma")
for a, b in LIMBS:
    if ok[a] and ok[b]:
        plt.plot([kp[a, 0], kp[b, 0]], [kp[a, 1], kp[b, 1]], c="cyan", lw=1)
plt.scatter(kp[ok, 0], kp[ok, 1], s=12, c="lime")
plt.title("packed sample 0: JHM targets + skeleton")
plt.axis("off")
plt.show()

Train: Adam lr 1e-3, batch 32, 20 epochs, lr halved at epochs 5/10/15 (the paper's config). After every epoch a checkpoint goes to `CKPT` and a quick validation PCK runs on ~1,700 held-out test frames, so you can watch whether the network is actually learning to localize (the thing to watch is PCK climbing, not just loss falling).

**Resuming after a disconnect:** set `resume = f"{CKPT}/epochNN.pt"` below (the highest NN you have) and rerun.

In [ ]:
import json
import torch
from torch.utils.data import Subset
from piw.eval import evaluate
from piw.train import train

assert torch.cuda.is_available(), "switch the runtime to GPU first"

val = Subset(test_ds, range(0, len(test_ds), 20))
val_log = []

def on_epoch_end(model, epoch):
    res = evaluate(model, val, device="cuda", batch=256, workers=2)
    val_log.append({"epoch": epoch, "overall": res["overall"], **res["groups"]})
    with open("val_log.json", "w") as f:
        json.dump(val_log, f, indent=1)
    print(f"  [val] epoch {epoch}  PCK@0.2 {res['overall']:.3f}  "
          + "  ".join(f"{g} {v:.3f}" for g, v in res["groups"].items()))

resume = None   # e.g. f"{CKPT}/epoch07.pt" after a disconnect
model, history = train(train_ds, epochs=20, batch=32, lr=1e-3,
                       device="cuda", workers=2, ckpt=CKPT, log_every=500,
                       on_epoch_end=on_epoch_end, resume=resume)

Final evaluation on the full official test split (33,753 frames), overall, per body-part group, and per joint. The paper's sanity signal: head joints should score worst (small parts scatter 12.5 cm radio waves poorly).

In [ ]:
res = evaluate(model, test_ds, device="cuda", batch=256, workers=2)
print(f"PCK@0.2 overall: {res['overall']:.4f}\n")
for g, v in res["groups"].items():
    print(f"  {g:>5}: {v:.4f}")
print("\nper joint, worst to best:")
for name, v in sorted(res["per_joint"].items(), key=lambda kv: kv[1]):
    print(f"  {name:>12} {v:.3f}")
with open("results_test.json", "w") as f:
    json.dump(res, f, indent=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(range(1, len(history) + 1), history)
axes[0].set(title="train loss", xlabel="epoch")
if val_log:
    axes[1].plot([r["epoch"] for r in val_log],
                 [r["overall"] for r in val_log])
axes[1].set(title="validation PCK@0.2", xlabel="epoch")
fig.tight_layout()
fig.savefig("training_curves.png", dpi=140)
plt.show()

In [ ]:
import shutil
for f in ["results_test.json", "val_log.json", "training_curves.png"]:
    if os.path.exists(f) and CKPT.startswith("/content/drive"):
        shutil.copy(f, CKPT)
print("saved to", CKPT, ":", sorted(os.listdir(CKPT)))

## Bringing the results home

Download (or grab from Drive) `results_test.json`, `val_log.json`, `training_curves.png`, and the final `epoch20.pt`, and add them to the repo so the README's stage 4 section can report real numbers.

Honest expectations, from the paper itself: in-domain PCK in a reasonable range with head joints worst; and if you ever evaluate on rooms or people the model never saw, expect a large drop. That collapse is the field's known open problem, not a bug here.